## Hello Nextmap!

### How to Setup

`Nextmap` supports a pure Python backend and a C++ backend called `emapcc` which is more efficient for large designs.

To enable `emapcc` backend, please install `pybind11` and build it with the following command:

In [ ]:
!bash build_emapcc.sh

`Nextmap` uses `gurobipy` as the default ILP solver. If you need to work on really large designs, please acquire a license from `Gurobi`: https://www.gurobi.com/academia/academic-program-and-licenses/.

In [ ]:
%pip install gurobipy
%pip install scipy
%pip install sqlite3

### How to Customize Your Own Equality Saturation for RTL

- Step 0: `Nextmap` takes Yosys JSON format as input. Take `tests/dot_product.v` as an example. Run the following Yosys commands to generate the JSON file:

In [ ]:
!yosys -p "read_verilog tests/dot_product.v; proc; opt_merge; opt_clean; write_json dot_product.json"

- Step 1: Define your cost model:

In [1]:
def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

- Step 2: Define your rewrite ruleset:

- Step 3: Define your technology library:

- Step 4: All done. You are ready to run `Nextmap`!

In [2]:
import emap
import json

TEST_NAME = "dot_product"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_right_matches = emap.rewrites.ematch_assoc_to_right(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_left_matches = emap.rewrites.ematch_assoc_to_left(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_assoc_to_right(netlist, assoc_to_right_matches)
    cnt += emap.rewrites.apply_assoc_to_left(netlist, assoc_to_left_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    if cnt == 0:
        print("No rewrites applied, stopping")
    netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, OutputFlag=False)

with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

Found 5 cells
Processing cell 0/5: $add$tests/dot_product.v:11$4
Database built with 226 wires and global clock 2
Applied 3 rewrites
No rewrites applied, stopping
C++ backend emapcc not found
Removed 3 dominated cells, 3 remain
C++ backend emapcc not found
Grouped 228 wires into 8 groups
Set parameter Username
Set parameter LicenseID to value 2690590
Academic license - for non-commercial use only - expires 2026-07-24
ILP model solved with objective value: 608.0


### Current Limitations

- `Nextmap` does not support blackbox instances (in progress);
- `Nextmap` does not support multiple clock domains (in progress);
- `Nextmap` does not support acyclic extraction (can be done if necessary).